# Surface-factor development feasibility

Development-only mechanical checks before freezing the blinded post-result evaluator. This notebook reads 2023–2024 option files only. It reports QC-state coverage and first-valid-quarter reconstruction, delta/vega stability and self-financing residuals. It does not display aggregate development profitability.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.forecasting import DEV_START, DEV_END
from src.time_series import build_daily_time_series
from src.surface_factor_trade import development_sanity

spx = pd.read_csv(ROOT / 'data/raw/spx_security_prices.csv', parse_dates=['date'])
spx = spx.loc[spx.date.between(DEV_START, DEV_END)].copy()
metrics = pd.read_csv(ROOT / 'data/processed/spx_skew_metrics_2023_2025.csv', parse_dates=['quote_date'])
metrics = metrics.loc[metrics.quote_date.between(DEV_START, DEV_END)].copy()
daily = build_daily_time_series(metrics, spx)
assert daily.quote_date.le(DEV_END).all()
assert spx.date.le(DEV_END).all()
raw_files = [ROOT / f'data/raw/spx_option_prices_{year}.csv' for year in (2023, 2024)]

## Coverage and mechanical examples

The calendar includes every development SPX session. RR25 values have already been masked by their individual QC flags. Examples are the first date with valid 30D and 60D states in each quarter, plus its next session; no date is selected using returns or P&L. Reconstruction must agree with the stored development states.

In [2]:
sanity = development_sanity(raw_files, daily, spx)
display(sanity['coverage'].to_frame())
display(sanity['mechanics'])
output = ROOT / 'data/processed/surface_factor_development'
output.mkdir(parents=True, exist_ok=True)
sanity['mechanics'].to_csv(output / 'mechanics.csv', index=False)
sanity['diagnostics'].to_csv(output / 'production_diagnostics.csv', index=False)

,count_or_fraction
development_sessions,502.000000
valid_30d_rr_states,500.000000
valid_60d_rr_states,498.000000
both_rr_states,496.000000
missing_state_fraction,0.011952


,quote_date,next_session,basket_available,accounting_available,reason,max_rr_state_difference,gross_vega,net_vega,max_delta_error,min_vega,max_abs_quantity,accounting_residual,liquidation_residual,min_aged_days
0,2023-01-04,2023-01-05,True,True,,5.551115e-17,1.0,0.000000e+00,2.157219e-12,35027.127526,0.000007,8.673617e-19,8.673617e-19,29.0
1,2023-04-03,2023-04-04,True,True,,9.020562e-17,1.0,2.775558e-17,2.481904e-13,37541.425462,0.000007,3.773024e-17,3.773024e-17,29.0
2,2023-07-03,2023-07-05,True,True,,1.387779e-17,1.0,0.000000e+00,1.620815e-12,40570.868835,0.000006,1.864828e-17,1.864828e-17,28.0
3,2023-10-02,2023-10-03,True,True,,4.857226e-17,1.0,2.775558e-17,1.698641e-13,39016.931880,0.000006,-5.290907e-17,-5.290907e-17,29.0
4,2024-01-02,2024-01-03,True,True,,2.081668e-17,1.0,0.000000e+00,4.994560e-12,43212.331846,0.000006,2.602085e-18,2.602085e-18,29.0
5,2024-04-01,2024-04-02,True,True,,9.367507e-17,1.0,0.000000e+00,1.942890e-14,47737.061850,0.000005,-6.505213e-18,-6.505213e-18,29.0
6,2024-07-01,2024-07-02,True,True,,4.857226e-17,1.0,0.000000e+00,1.456613e-13,49856.776120,0.000005,-2.602085e-18,-2.602085e-18,29.0
7,2024-10-01,2024-10-02,True,True,,6.245005e-17,1.0,0.000000e+00,5.457856e-13,51989.700008,0.000005,5.307170e-17,5.307170e-17,29.0


## Numerical checks

We expect forward-delta distance near zero, gross vega one, net vega zero and self-financing residuals near machine precision. Unsupported states are reported rather than substituted. Only residuals are shown; no aggregate development P&L is constructed or displayed.

In [3]:
mechanics = sanity['mechanics']
summary = pd.Series({
    'mechanical_examples': len(mechanics),
    'available_baskets': int(mechanics.basket_available.sum()),
    'available_accounting_checks': int(mechanics.accounting_available.sum()),
    'maximum_delta_error': mechanics.get('max_delta_error', pd.Series(dtype=float)).max(),
    'maximum_gross_vega_error': (mechanics.get('gross_vega', pd.Series(dtype=float)) - 1).abs().max(),
    'maximum_abs_net_vega': mechanics.get('net_vega', pd.Series(dtype=float)).abs().max(),
    'maximum_abs_accounting_residual': mechanics.get('accounting_residual', pd.Series(dtype=float)).abs().max(),
    'maximum_abs_liquidation_residual': mechanics.get('liquidation_residual', pd.Series(dtype=float)).abs().max(),
    'maximum_rr_reconstruction_error': mechanics.max_rr_state_difference.max(),
}, name='mechanical_check')
display(summary.to_frame())
assert summary['maximum_gross_vega_error'] < 1e-10
assert summary['maximum_abs_net_vega'] < 1e-10
assert summary['maximum_abs_accounting_residual'] < 1e-10
assert summary['maximum_abs_liquidation_residual'] < 1e-10
print('Development mechanics checked; new 2025 factor/portfolio results have not been opened.')

,mechanical_check
mechanical_examples,8.000000e+00
available_baskets,8.000000e+00
available_accounting_checks,8.000000e+00
maximum_delta_error,4.994560e-12
maximum_gross_vega_error,0.000000e+00
maximum_abs_net_vega,2.775558e-17
maximum_abs_accounting_residual,5.307170e-17
maximum_abs_liquidation_residual,5.307170e-17
maximum_rr_reconstruction_error,9.367507e-17


Development mechanics checked; new 2025 factor/portfolio results have not been opened.
